In [1]:
#Directory 
directory = "C:\\Users\\20234503\\Desktop\\Research\\Weather and Strikes\\Project\\"

#Packages
import pandas as pd

# --- keep this near your imports ---
import types
import inspect
import sys

def clear_env(keep=None):
    """
    Delete user-defined globals while keeping:
      - all imported modules (e.g. pd, np, gpd, plt, string, etc.)
      - functions/classes imported via `from ... import ...` (e.g. unary_union, Counter, Patch)
      - anything explicitly listed in `keep`
      - underscore-prefixed names and this function itself
    """
    if keep is None:
        keep = []
    keep = set(keep) | {"clear_env"}

    g = globals()
    for name, val in list(g.items()):
        # never touch dunders, explicit keepers, or this function
        if name.startswith("_") or name in keep:
            continue

        # keep modules (e.g., pd, np, gpd, plt, string, re, json, sys, types, etc.)
        if isinstance(val, types.ModuleType):
            continue

        # keep imported functions/classes (e.g., unary_union, Counter, Patch, product)
        # (callables or types) whose defining module is not __main__
        if (callable(val) or isinstance(val, type)):
            mod = inspect.getmodule(val)
            if mod is not None and mod.__name__ != "__main__":
                continue

        # otherwise, remove it (likely user-created data/vars/objects)
        try:
            del g[name]
        except Exception:
            pass

# Checked if code is correct 24/09/2025 

In [4]:
############################################################
# Matching Labour Unrest and Weather Data (Multi-location) #
############################################################
# (A) Matching of Labour Unrest and Weather
Weather = pd.read_csv(directory+"Data\\weather.csv")
Labour_Unrest = pd.read_csv(directory+"Data\\labour_unrest_county_day_panel.csv")   
print 
Weather_Labour_Unrest = Weather.merge(Labour_Unrest, on=['FIPS', 'date']) #Fix the ISSUE WITH A LEAP YEAR
# Checks
duplicates = Weather[Weather.duplicated(subset=['FIPS', 'date'], keep=False)]
duplicate_counts = duplicates.groupby(['FIPS', 'date']).size().reset_index(name='count')
print(duplicate_counts)

print("Weather date range:")
print("  Min:", Weather["date"].min())
print("  Max:", Weather["date"].max())

print("\nLabour Unrest date range:")
print("  Min:", Labour_Unrest["date"].min())
print("  Max:", Labour_Unrest["date"].max())

# (B) Matching of Auxilary Data
CBP = pd.read_csv(directory + 'Data\\Raw\\CBP\\cbp21co.txt' )
#Union = pd.read_csv(directory + 'Data\\Raw\\Union\\State_Union_Coverage_2021.csv' )

# Cleaning CBP Data
CBP = CBP[CBP['naics'] == '------']
selected_columns = [ 
    'fipstate', 'fipscty', 'emp', 'est'
]
CBP = CBP[selected_columns]
CBP['FIPS'] = CBP['fipstate'].astype(str).str.zfill(2) + CBP['fipscty'].astype(str).str.zfill(3)
CBP['FIPS'] = CBP['FIPS'].astype('Int64')

# Merging Data
# Aux_Data = CBP.merge(
  # Union,
  # on=['fipstate'],
  # how='left'
#)

Weather_Labour_Unrest = Weather_Labour_Unrest.merge(
   CBP,
   on=['FIPS'],
   how='left'
)

# (C) Filter Dataset to only Counties with at least one event
Weather_Labour_Unrest_Event =  Weather_Labour_Unrest[Weather_Labour_Unrest['at least one event'] == 1]
#Check Min and Max Dates
print("Min Date: ", Weather_Labour_Unrest_Event['date'].min())    
print("Max Date: ", Weather_Labour_Unrest_Event['date'].max())

# (D) Exporting Dataset
Weather_Labour_Unrest_Event.to_csv(directory+"Data\\Weather_Labour_Unrest_Event.csv", index=False)
# Checked if code is correct 24/09/2024 
clear_env(keep=["directory", "Weather_Labour_Unrest_Event", "Labour_Unrest"])

Empty DataFrame
Columns: [FIPS, date, count]
Index: []
Weather date range:
  Min: 2020-01-01
  Max: 2025-10-31

Labour Unrest date range:
  Min: 2021-01-01
  Max: 2025-12-27
Min Date:  2021-01-01
Max Date:  2025-10-31


In [3]:
#############################################################
# Matching Labour Unrest and Weather Data (Single-location) #
#############################################################
# (A) Matching of Labour Unrest and Weather
Weather = pd.read_csv(directory+"Data\\weather.csv")
Labour_Unrest = pd.read_csv(directory+"Data\\labour_unrest_county_day_panel_sl.csv")
Weather_Labour_Unrest = Weather.merge(Labour_Unrest, on=['FIPS', 'date']) #Fix the ISSUE WITH A LEAP YEAR
# Checks
duplicates = Weather[Weather.duplicated(subset=['FIPS', 'date'], keep=False)]
duplicate_counts = duplicates.groupby(['FIPS', 'date']).size().reset_index(name='count')
print(duplicate_counts)

# (B) Matching of Auxilary Data
CBP = pd.read_csv(directory + 'Data\\Raw\\CBP\\cbp21co.txt' )
#Union = pd.read_csv(directory + 'Data\\Raw\\Union\\State_Union_Coverage_2021.csv' )

# Cleaning CBP Data
CBP = CBP[CBP['naics'] == '------']
selected_columns = [ 
    'fipstate', 'fipscty', 'emp', 'est'
]
CBP = CBP[selected_columns]
CBP['FIPS'] = CBP['fipstate'].astype(str).str.zfill(2) + CBP['fipscty'].astype(str).str.zfill(3)
CBP['FIPS'] = CBP['FIPS'].astype('Int64')

# Merging Data
#Aux_Data = CBP.merge(
   #Union,
   #on=['fipstate'],
   #how='left'
#)

Weather_Labour_Unrest = Weather_Labour_Unrest.merge(
   CBP,
   on=['FIPS'],
   how='left'
)

# (C) Filter Dataset to only Counties with at least one event
Weather_Labour_Unrest_Event =  Weather_Labour_Unrest[Weather_Labour_Unrest['at least one event'] == 1]
#Check Min and Max Dates
print("Min Date: ", Weather_Labour_Unrest_Event['date'].min())    
print("Max Date: ", Weather_Labour_Unrest_Event['date'].max())

# (D) Exporting Dataset
Weather_Labour_Unrest_Event.to_csv(directory+"Data\\Weather_Labour_Unrest_Event_sl.csv", index=False)
# Checked if code is correct 24/09/2024 
clear_env(keep=["directory", "Weather_Labour_Unrest_Event_sl"])

Empty DataFrame
Columns: [FIPS, date, count]
Index: []
Min Date:  2021-01-01
Max Date:  2024-10-14


In [4]:
#############################################################
# Matching Labour Unrest and Weather Data (Only Counting One Multi-location per County) #
#############################################################
# (A) Matching of Labour Unrest and Weather
Weather = pd.read_csv(directory+"Data\\weather.csv")
Labour_Unrest = pd.read_csv(directory+"Data\\labour_unrest_county_day_panel_nodup_cd.csv")
Weather_Labour_Unrest = Weather.merge(Labour_Unrest, on=['FIPS', 'date']) #Fix the ISSUE WITH A LEAP YEAR
# Checks
duplicates = Weather[Weather.duplicated(subset=['FIPS', 'date'], keep=False)]
duplicate_counts = duplicates.groupby(['FIPS', 'date']).size().reset_index(name='count')
print(duplicate_counts)

# (B) Matching of Auxilary Data
CBP = pd.read_csv(directory + 'Data\\Raw\\CBP\\cbp21co.txt' )
#Union = pd.read_csv(directory + 'Data\\Raw\\Union\\State_Union_Coverage_2021.csv' )

# Cleaning CBP Data
CBP = CBP[CBP['naics'] == '------']
selected_columns = [ 
    'fipstate', 'fipscty', 'emp', 'est'
]
CBP = CBP[selected_columns]
CBP['FIPS'] = CBP['fipstate'].astype(str).str.zfill(2) + CBP['fipscty'].astype(str).str.zfill(3)
CBP['FIPS'] = CBP['FIPS'].astype('Int64')

# Merging Data
#Aux_Data = CBP.merge(
   #Union,
   #on=['fipstate'],
   #how='left'
#)

Weather_Labour_Unrest = Weather_Labour_Unrest.merge(
   CBP,
   on=['FIPS'],
   how='left'
)

# (C) Filter Dataset to only Counties with at least one event
Weather_Labour_Unrest_Event =  Weather_Labour_Unrest[Weather_Labour_Unrest['at least one event'] == 1]
#Check Min and Max Dates
print("Min Date: ", Weather_Labour_Unrest_Event['date'].min())    
print("Max Date: ", Weather_Labour_Unrest_Event['date'].max())

# (D) Exporting Dataset
Weather_Labour_Unrest_Event.to_csv(directory+"Data\\Weather_Labour_Unrest_Event_nodup_cd.csv", index=False)
# Checked if code is correct 24/09/2024 
clear_env(keep=["directory", "Weather_Labour_Unrest_Event_sl"])

Empty DataFrame
Columns: [FIPS, date, count]
Index: []
Min Date:  2021-01-01
Max Date:  2024-10-14


In [3]:
#############################################################
# Matching Labour Unrest and Weather Data (Event-Level Weighting) #
#############################################################
# (A) Matching of Labour Unrest and Weather
Weather = pd.read_csv(directory+"Data\\weather.csv")
Labour_Unrest = pd.read_csv(directory+"Data\\labour_unrest_county_day_panel_event_weighted.csv")
Weather_Labour_Unrest = Weather.merge(Labour_Unrest, on=['FIPS', 'date']) #Fix the ISSUE WITH A LEAP YEAR
# Checks
duplicates = Weather[Weather.duplicated(subset=['FIPS', 'date'], keep=False)]
duplicate_counts = duplicates.groupby(['FIPS', 'date']).size().reset_index(name='count')
print(duplicate_counts)

# (B) Matching of Auxilary Data
CBP = pd.read_csv(directory + 'Data\\Raw\\CBP\\cbp21co.txt' )
#Union = pd.read_csv(directory + 'Data\\Raw\\Union\\State_Union_Coverage_2021.csv' )

# Cleaning CBP Data
CBP = CBP[CBP['naics'] == '------']
selected_columns = [ 
    'fipstate', 'fipscty', 'emp', 'est'
]
CBP = CBP[selected_columns]
CBP['FIPS'] = CBP['fipstate'].astype(str).str.zfill(2) + CBP['fipscty'].astype(str).str.zfill(3)
CBP['FIPS'] = CBP['FIPS'].astype('Int64')

# Merging Data
#Aux_Data = CBP.merge(
   #Union,
   #on=['fipstate'],
   #how='left'
#)

Weather_Labour_Unrest = Weather_Labour_Unrest.merge(
   CBP,
   on=['FIPS'],
   how='left'
)

# (C) Filter Dataset to only Counties with at least one event
Weather_Labour_Unrest_Event =  Weather_Labour_Unrest[Weather_Labour_Unrest['at least one event'] == 1]
#Check Min and Max Dates
print("Min Date: ", Weather_Labour_Unrest_Event['date'].min())    
print("Max Date: ", Weather_Labour_Unrest_Event['date'].max())

# (D) Exporting Dataset
Weather_Labour_Unrest_Event.to_csv(directory+"Data\\Weather_Labour_Unrest_Event_Weighted.csv", index=False)

Empty DataFrame
Columns: [FIPS, date, count]
Index: []
Min Date:  2021-01-01
Max Date:  2024-10-14
